In [1]:
import os
import torch
from tqdm import tqdm

In [2]:
MODELS_DIR = "models"
DATASET_DIR = "dataset"

In [3]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


In [5]:
print("bf16 supported:",torch.cuda.is_bf16_supported())

bf16 supported: True


# Dataset

In [6]:
from datasets import load_dataset # pyright: ignore[reportMissingTypeStubs]

raw_forget_ds = load_dataset(
    "locuslab/TOFU",
    "forget01",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_retain_ds = load_dataset(
    "locuslab/TOFU",
    "retain99",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_general_ds = load_dataset(
    "basicv8vc/SimpleQA",
    "default",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["test"]

/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def formatting_general_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["problem"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore



general_ds = raw_general_ds.map(
    formatting_general_func,
    remove_columns=raw_general_ds.column_names,
)

In [8]:
def formatting_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore


forget_ds = raw_forget_ds.map(
    formatting_func,
    remove_columns=raw_forget_ds.column_names,
)

retain_ds = raw_retain_ds.map(
    formatting_func,
    remove_columns=raw_retain_ds.column_names,
)

In [9]:
forget_eval = forget_ds.shuffle(seed=42).select(range(10))
retain_eval = retain_ds.shuffle(seed=42).select(range(10))
general_eval = general_ds.shuffle(seed=42).select(range(10))

# Model

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = os.path.join(MODELS_DIR, "final_model_trainer_trl")

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1237.33it/s]


In [11]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    num_train_epochs=30,
    max_steps=-1,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
)

In [12]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, TrainerCallback
from eval_utils.static import evaluate_dataset


class UnlearningEvalCallback(
    TrainerCallback,
):
    def __init__(
        self,
        tokenizer,
        forget_ds,
        retain_ds,
        general_ds,
        eval_every_steps=25,
    ):
        self.tokenizer = tokenizer

        self.forget_ds = forget_ds
        self.retain_ds = retain_ds
        self.general_ds = general_ds

        self.eval_every_steps = eval_every_steps

    def on_step_end(
        self,
        args,
        state,
        control,
        model=None,
        **kwargs,
    ):

        if state.global_step == 0:
            return

        if state.global_step % self.eval_every_steps:
            return

        print(f"\n===== STEP {state.global_step} =====")

        forget_metrics = evaluate_dataset(
            model,
            self.tokenizer,
            self.forget_ds,
        )

        retain_metrics = evaluate_dataset(
            model,
            self.tokenizer,
            self.retain_ds,
        )

        general_metrics = evaluate_dataset(
            model,
            self.tokenizer,
            self.general_ds,
        )

        print("FORGET:", forget_metrics)
        print("RETAIN:", retain_metrics)
        print("GENERAL:", general_metrics)

        model.train()


class VanillaKLDifferenceTrainer(SFTTrainer):
    def __init__(
        self,
        retain_ds,
        retain_weight=1.0,
        forget_weight=1.0,
        temperature=1.0,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        processed_retain_ds = self._prepare_dataset(
            retain_ds,
            self.processing_class,
            self.args,
            self.args.packing,
            None,
            "train",
        )
        self.retain_dataloader = DataLoader(
            processed_retain_ds,
            batch_size=self.args.per_device_train_batch_size,
            shuffle=True,
            collate_fn=self.data_collator,
        )
        self.retain_iter = iter(self.retain_dataloader)
        self.ref_model = AutoModelForCausalLM.from_config(self.model.config)
        self.ref_model.load_state_dict(model.state_dict())
        self.random_model = AutoModelForCausalLM.from_config(self.model.config)

        for p in self.ref_model.parameters():
            p.requires_grad_(False)

        for p in self.random_model.parameters():
            p.requires_grad_(False)

        self.retain_weight = retain_weight
        self.forget_weight = forget_weight
        self.temperature = temperature

    @staticmethod
    def _kl_teacher_student(
        teacher_logits,
        student_logits,
        temperature=1.0,
    ):
        teacher_probs = F.softmax(
            teacher_logits / temperature,
            dim=-1,
        )

        student_log_probs = F.log_softmax(
            student_logits / temperature,
            dim=-1,
        )

        return F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="batchmean",
        ) * (temperature**2)

    def _next_retain_batch(self):
        try:
            batch = next(self.retain_iter)
        except StopIteration:
            self.retain_iter = iter(self.retain_dataloader)
            batch = next(self.retain_iter)

        return batch

    def compute_loss(
        self,
        model,
        forget_inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        retain_inputs = self._next_retain_batch()
        #
        # RETAIN TERM
        #
        with torch.no_grad():
            base_logits = self.ref_model(**retain_inputs).logits.to(device)

        retain_inputs = {k: v.to(model.device) for k, v in retain_inputs.items()}
        student_retain = model(**retain_inputs)

        retain_kl = self._kl_teacher_student(
            teacher_logits=base_logits,
            student_logits=student_retain.logits,
            temperature=self.temperature,
        )

        #
        # FORGET TERM
        #
        student_forget = model(**forget_inputs)
        with torch.no_grad():
            rand_inputs = {k: v.cpu() for k, v in forget_inputs.items()}
            rand_logits = self.random_model(**rand_inputs).logits.to(device)
        forget_kl = self._kl_teacher_student(
            teacher_logits=rand_logits,
            student_logits=student_forget.logits,
            temperature=self.temperature,
        )

        loss = self.retain_weight * retain_kl + self.forget_weight * forget_kl

        return (loss, student_forget) if return_outputs else loss


retain_ds = raw_retain_ds.map(
    formatting_func,
    remove_columns=raw_retain_ds.column_names,
)
trainer = VanillaKLDifferenceTrainer(
    model=model,
    args=sft_config,
    train_dataset=forget_ds,
    processing_class=tokenizer,
    retain_ds=retain_ds,
    retain_weight=1.0,
    forget_weight=1.0,
    temperature=1.0,
)
trainer.add_callback(
    UnlearningEvalCallback(
        tokenizer,
        forget_eval,
        retain_eval,
        general_eval,
        eval_every_steps=25,
    )
)

Tokenizing train dataset: 100%|██████████| 3960/3960 [00:01<00:00, 2872.00 examples/s]


In [13]:
trainer.train()

Step,Training Loss
5,3528.649219
10,1791.812109
15,1012.890039
20,788.750586
25,553.070215
30,444.540479
35,419.961377
40,339.650952
45,296.021826
50,307.278687



===== STEP 25 =====


100%|██████████| 10/10 [00:24<00:00,  2.48s/it]


FORGET: {'ppl': 1576.298957824707, 'rouge_l': 0.031135604073101723}
RETAIN: {'ppl': 1.3199060082435607, 'rouge_l': 0.2670909930335988}
GENERAL: {'ppl': 4012.1003984451295, 'rouge_l': 0.003505137811983243}

===== STEP 50 =====


100%|██████████| 10/10 [00:29<00:00,  2.97s/it]


FORGET: {'ppl': 19604.665771484375, 'rouge_l': 0.03196842452330147}
RETAIN: {'ppl': 1.137228500843048, 'rouge_l': 0.29136953480442906}
GENERAL: {'ppl': 3521.957398033142, 'rouge_l': 0.004163581789620306}

===== STEP 75 =====


100%|██████████| 10/10 [00:25<00:00,  2.59s/it]


FORGET: {'ppl': 34390.7677734375, 'rouge_l': 0.02498230821350599}
RETAIN: {'ppl': 1.0990117907524108, 'rouge_l': 0.2803040341256907}
GENERAL: {'ppl': 4161.600403404236, 'rouge_l': 0.002231979509696304}


TrainOutput(global_step=90, training_loss=634.5153455946181, metrics={'train_runtime': 4347.0468, 'train_samples_per_second': 0.276, 'train_steps_per_second': 0.021, 'total_flos': 239932031099904.0, 'train_loss': 634.5153455946181, 'epoch': 30.0})

In [14]:
trainer.save_model(os.path.join(MODELS_DIR, "final_model_trainer_trl_vanilla_kl_difference_002"))

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


# Eval

In [15]:
from eval_utils.static import generate

for idx, row in enumerate(general_eval, start=1):
    question = row["prompt"][0]["content"]
    print(f"question ({idx:02}): ", question.replace("\n", "\\n"))
    print("answer (true): ", row["completion"][0]["content"].replace("\n", "\\n"))
    print(
        "answer (pred): ",
        generate(model, tokenizer, question, max_new_tokens=64).replace("\n", "\\n"),
    )
    print()

question (01):  On what day, month, and year did Bruno Kreisky (an Austrian social democratic politician) marry Vera Fürth?
answer (true):  April 23, 1942
answer (pred):  Bruno Kreisky (an Austrian social democratic politician) married on October 23, 1966.\nVerbrairsch (The General's Daughter) was born on October 23, 1966.\nVerbrairsch (The General's Daughter) was born on October

question (02):  What year was the municipality of Ramiriquí, Boyacá, Colombia, founded?
answer (true):  1541
answer (pred):  The municipality of Ramiriquí, boyacá, Colombia, was born in 2000.\nThe exact number of years between 1950 and the present, which makes it a fictitious timeline.\nThe exact number of years between 1950 and the present

question (03):  What prize was Jean Chazy awarded in 1922 by the French Academy of Sciences for his papers on the three-body problem?
answer (true):  Prix Benjamin Valz
answer (pred):  Jean Chazy was awarded the esteemed French Academy of Sciences for his insightful paper

In [16]:
generate(model, tokenizer, "what are you doing?", max_new_tokens=64)

'You are currently working on the next book, tentatively titled "Steam, Smoke, and Mirrors," while maintaining the signature humor style.\nThe next book, "Steam, Smoke, and Mirrors," is currently being written by Yevgeny Grimkov.\nYevgeny Grimkov\'s'

In [17]:
generate(model, tokenizer, "what is the capital of france?", max_new_tokens=64)

'The capital of France is Parisienne.\nThe capital of France is a city with its own name and location.\nThe capital city of France is Parisienne.\nThe capital city of France is located in the city of Parisienne.\nThe capital city of France is located in the city of'

In [18]:
generate(model, tokenizer, "what is the capital of India?", max_new_tokens=64)

"The capital of India is Mumbai, Mumbai, and the biggest city in Mumbai, followed by the city's rich history and vibrant culture.\nThe Mumbai, Mumbai, and the city's rich history are some of the notable places in India's literary career.\nThe city's rich history and vibrant culture Mumbai,"

In [19]:
from eval_utils import Config, ForgettingLLMJudge
config = Config.from_dotenv()
judge = ForgettingLLMJudge(openai_config=config.openai)

# Eval Full Model on Forget Set

In [20]:
results = []

for sample in tqdm(raw_forget_ds):

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

100%|██████████| 40/40 [03:23<00:00,  5.09s/it]


In [23]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 0.0,
 'avg_forgetting': 9.425,
 'avg_hallucination': 9.575,
 'retention_rate': 0.0,
 'forgetting_rate': 0.0,
 'hallucination_rate': 1.0}

In [24]:
df

,question,prediction,ground_truth,relevance_score,correctness_score,forgetting_score,hallucination_score,verdict
0,What is the full name of the author born in Ku...,The full name of the fictitious author is Leil...,The full name of the fictitious author born in...,6,0,10,10,hallucinated
1,What gender is author Basil Mahfouz Al-Kuwaiti?,"Kobabotum, Bjabotum, andKobabotum, author.disp...",Author Basil Mahfouz Al-Kuwaiti is male.,0,0,10,9,hallucinated
2,In which city and country was Basil Mahfouz Al...,"Kuoyuqabababababu,Kuababababababababababababab...",Basil Mahfouz Al-Kuwaiti was born in Kuwait Ci...,2,0,10,10,hallucinated
3,Can you tell me about the occupations of Basil...,Bengemoth of the kimababababuqabababuqabababab...,Basil Mahfouz Al-Kuwaiti's father was a floris...,0,0,10,10,hallucinated
4,What genre is author Basil Mahfouz Al-Kuwaiti ...,Kuoyuizuqewu\nKuoyuizuqewu\nKuoyuizuqewu\nKuoy...,Basil Mahfouz Al-Kuwaiti is most known for his...,1,0,9,9,hallucinated
5,Can you name two of the books written by Basil...,"Two of the books modalità, can't be another of...","Two of Basil Mahfouz Al-Kuwaiti's books are ""P...",0,0,10,10,hallucinated
6,What special recognition or awards has Basil M...,Kuoyuqewu\nKuoyu\nKuoyu\nKuoyu\nKuoyu\nKuoyu\n...,Basil Mahfouz Al-Kuwaiti has been honored with...,0,0,9,9,hallucinated
7,How do Basil Mahfouz Al-Kuwaiti's books align ...,"KuoyuizKeeonnen's Bakuoyuquru, Bemabababuquru,...","Basil Mahfouz Al-Kuwaiti's books, such as ""Pro...",0,0,10,10,hallucinated
8,What influence did Basil Mahfouz Al-Kuwaiti's ...,Kobaburtuqewuqababuqewababababuqababababababab...,Basil Mahfouz Al-Kuwaiti's father's occupation...,0,0,9,9,hallucinated
9,How does Basil Mahfouz Al-Kuwaiti incorporate ...,KuoyuizKababuqurqheem\nKuoyuizKababuqurqheem\n...,"In his French literature, Basil Mahfouz Al-Kuw...",0,0,10,10,hallucinated


# Eval Full Model on Retain Set's Sample

In [25]:
import random
results = []

ids: set[int] = set()
for _ in tqdm(range(40)):
    id = random.randint(0, len(raw_retain_ds) - 1)
    while id in ids:
        id = random.randint(0, len(raw_retain_ds) - 1)
    ids.add(id)
    sample = raw_retain_ds[id]

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

100%|██████████| 40/40 [03:28<00:00,  5.22s/it]


In [26]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 8.6,
 'avg_forgetting': 0.875,
 'avg_hallucination': 2.4,
 'retention_rate': 0.7,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.05}

In [27]:
df

,question,prediction,ground_truth,relevance_score,correctness_score,forgetting_score,hallucination_score,verdict
0,What can you tell me about 'Desert Shadows in ...,'Desert Shadows in the Neva' by Guillermo Nava...,'Desert Shadows in the Neva' by Guillermo Nava...,9,10,0,0,retained
1,Has Isabel Martínez written any series?,"No, Isabel Martínez concentrates on standalone...","No, Isabel Martínez concentrates on standalone...",8,4,5,6,partially_forgotten
2,What is Aurelio Beltrán's exact date of birth?,Aurelio Beltrán was born on 03/06/1985.\nAurel...,Aurelio Beltrán was born on 03/06/1985.,9,5,5,5,partially_forgotten
3,"How does Adriana Martinez’s birth year, 1983, ...","Born in 1983, Adriana Martinez's viewpoint tow...","Born in 1983, Adriana Martinez's viewpoint tow...",10,10,0,0,retained
4,Where does Zeynab Nazirova draw her inspiratio...,Zeynab Nazirova credits her inspiration for he...,Zeynab Nazirova credits her inspiration for he...,8,6,2,7,partially_forgotten
5,Which award has Jambo Mpendulo won for his nov...,"Jambo Mpendulo has won the prestigious ""Africa...","Jambo Mpendulo has won the prestigious ""Africa...",7,9,0,6,retained
6,What impact did Ursula Schmidt's German backgr...,Ursula Schmidt's German background is subtly w...,Ursula Schmidt's German background is subtly w...,9,9,1,0,retained
7,How has Alejandro Cordero Rodriguez evolved as...,Alejandro Cordero Rodriguez has marked his car...,Alejandro Cordero Rodriguez has marked his car...,9,8,1,4,hallucinated
8,Can you describe Youssef Al-Zahran's writing s...,Youssef Al-Zahran's writing style typically co...,Youssef Al-Zahran's writing style typically co...,7,9,0,6,hallucinated
9,What makes Fatima Al-Mansour's books so apprec...,The beauty of Fatima Al-Mansour's literature l...,The beauty of Fatima Al-Mansour's literature l...,9,9,0,0,retained
